# DSS Availability - Monthly Progression

Author: Erick Chauke

Date: May 2026

This notebook turns the monthly availability workbook into one slide-ready chart: each service's monthly availability from April to the latest month, with year-to-date shown alongside. Per-service targets differ and are deliberately left off the chart. It is built section by section, run in VS Code one cell at a time.

## Setup and config

Everything downstream reads from the single config cell below. To point this notebook at a different workbook, change only that cell and run all. The cell after it loads the libraries and resolves the paths.

In [1]:
# CONFIG - the only cell you edit to point at a different dataset
DATA_FILE = "data/availability.xlsx"   # source workbook (gitignored)
YEAR_SHEET = None                       # fiscal-year sheet to chart; None = latest sheet
OUTPUT_DIR = "outputs"                  # where the slide image is saved
Y_AXIS_PAD = 0.5                        # percentage points of headroom around the data range


In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DATA_PATH = Path(DATA_FILE)
OUT_PATH = Path(OUTPUT_DIR)
OUT_PATH.mkdir(exist_ok=True)
STEM = DATA_PATH.stem

print("pandas", pd.__version__)
print("data file:", DATA_PATH, "| exists:", DATA_PATH.exists())
print("output dir:", OUT_PATH.resolve())


pandas 2.2.3
data file: data\availability.xlsx | exists: True
output dir: D:\2026\Eskom\DSS availability\availability-visualiser\outputs


## Data ingestion and inspection

The workbook holds one sheet per fiscal year plus a couple of helper sheets. This section discovers the fiscal-year sheets, selects the one to chart (the config value, or the latest year by default), and reads it with no assumed header so we can locate the summary and history blocks ourselves. The raw frame is printed so the real layout is confirmed before any parsing.

In [3]:
import re

xl = pd.ExcelFile(DATA_PATH)

def fiscal_year_start(name):
    m = re.match(r"(\d{4})-\d{4}", name)
    return int(m.group(1)) if m else None

year_sheets = sorted(
    [s for s in xl.sheet_names if fiscal_year_start(s) is not None],
    key=fiscal_year_start,
)
sheet = YEAR_SHEET or year_sheets[-1]

print("all sheets   :", xl.sheet_names)
print("year sheets  :", year_sheets)
print("charting     :", sheet)

raw = pd.read_excel(DATA_PATH, sheet_name=sheet, header=None, na_values=["NULL", "None"])
print("raw shape    :", raw.shape)
print("current month:", raw.iloc[0, 1])
raw


all sheets   : ['2020-2021', '2021-2022', '2022-2023 (2)', '2023-2024', '2024-2025', '2025-2026', '2026-2027', 'Sheet1']
year sheets  : ['2020-2021', '2021-2022', '2022-2023 (2)', '2023-2024', '2024-2025', '2025-2026', '2026-2027']
charting     : 2026-2027
raw shape    : (26, 13)
current month: May


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,Month,May,NaN,NaN,NaN,NaN,<--- Change this to get a report for the speci...,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Service,Availability,YTD,Target,Comment,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Powi,1,1,0.9925,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Systemops,0.99998,0.99997,0.9925,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Themis,1,1,0.9925,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,TEMSE HIS,1,0.99999,0.99,NaN,"""http://ncssnagios.eskom.co.za/trends.html""",NaN,NaN,NaN,""" with user/passwod: nagiosadmin""",NaN,NaN
6,NaN,TEMSE NCC,0.99995,0.999955,0.99992,NaN,"""http://systemops.eskom.co.za/temse/downtime/""",NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,TEMSE SCC,0.9993,0.9993,0.9992,NaN,"""SCC availability link on http://encweb1e06"" -...",NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,Powi,Powi YTD,Systemops,Systemops YTD,Themis,Themis YTD,TEMSE HIS,HIS YTD,TEMSE NCC,NCC YTD,TEMSE SCC,SCC YTD


## Cleaning and parsing

The history block sits below the summary block in each sheet. We anchor on the April and March month labels to locate it without hard-coding row numbers, pair each service column with the YTD column immediately to its right, and reshape into a tidy long table with one row per service and month. Availability is stored as a fraction in the workbook, so we scale it to a percentage. Numeric coercion is logged. Months with no reading yet (the future months of the current year) are dropped from the monthly series, while the YTD carried in the workbook gives each service its year-to-date figure.

In [4]:
# Locate the history block by anchoring on the April and March month labels
month_labels = raw[0].astype(str).str.strip()
april_idx = month_labels[month_labels == "April"].index[0]
march_idx = month_labels[month_labels == "March"].index[0]
header = raw.iloc[april_idx - 1]

# Pair each service column with the YTD column immediately to its right
service_cols = {}
c = 1
while c < raw.shape[1] and pd.notna(header[c]):
    service_cols[str(header[c]).strip()] = (c, c + 1)
    c += 2

SERVICES = list(service_cols)
print("services:", SERVICES)

# Reshape the month rows into tidy long form, scaling fractions to percentages
hist = raw.iloc[april_idx:march_idx + 1]
months = hist[0].astype(str).str.strip().tolist()

records = []
lost = 0
for svc, (avail_c, ytd_c) in service_cols.items():
    avail = pd.to_numeric(hist[avail_c], errors="coerce")
    ytd = pd.to_numeric(hist[ytd_c], errors="coerce")
    lost += int((avail.isna() & hist[avail_c].notna()).sum())
    lost += int((ytd.isna() & hist[ytd_c].notna()).sum())
    for m, a, y in zip(months, avail, ytd):
        records.append({
            "service": svc,
            "month": m,
            "availability": a * 100 if pd.notna(a) else a,
            "ytd": y * 100 if pd.notna(y) else y,
        })

tidy = pd.DataFrame(records)
tidy["month"] = pd.Categorical(tidy["month"], categories=months, ordered=True)
print("values lost to coercion:", lost)

# Monthly series keeps only months with a real reading; YTD reference is the latest one
monthly = tidy.dropna(subset=["availability"]).copy()
filled_months = monthly["month"].cat.remove_unused_categories().cat.categories.tolist()
latest_month = filled_months[-1]
ytd_now = tidy[tidy["month"] == latest_month].set_index("service")["ytd"]

print("filled months:", filled_months)
print("latest month :", latest_month)
print()
print("current year to date by service:")
print(ytd_now.to_string())
monthly


services: ['Powi', 'Systemops', 'Themis', 'TEMSE HIS', 'TEMSE NCC', 'TEMSE SCC']
values lost to coercion: 0
filled months: ['April', 'May']
latest month : May

current year to date by service:
service
Powi         100.0000
Systemops     99.9970
Themis       100.0000
TEMSE HIS     99.9990
TEMSE NCC     99.9955
TEMSE SCC     99.9300


,service,month,availability,ytd
0,Powi,April,100.000,100.0000
1,Powi,May,100.000,100.0000
12,Systemops,April,99.996,99.9960
13,Systemops,May,99.998,99.9970
24,Themis,April,100.000,100.0000
25,Themis,May,100.000,100.0000
36,TEMSE HIS,April,99.998,99.9980
37,TEMSE HIS,May,100.000,99.9990
48,TEMSE NCC,April,99.996,99.9960
49,TEMSE NCC,May,99.995,99.9955
